In [8]:
import sqlite3
import pandas as pd
import requests

def fetch_world_bank_data(indicator_code, column_name):
    """Fetches data from the World Bank API for a specific indicator."""
    print(f"Fetching {column_name}...")
    url = f"http://api.worldbank.org/v2/country/all/indicator/{indicator_code}?date=2000:2023&format=json&per_page=10000"
    
    response = requests.get(url).json()
    
    if len(response) > 1 and response[1]:
        data = response[1]
        records = [
            {
                "country": item["country"]["value"],
                "country_code": item["countryiso3code"],
                "year": int(item["date"]),
                column_name: item["value"]
            }
            for item in data if item["countryiso3code"] != ""
        ]
        return pd.DataFrame(records)
    return pd.DataFrame()

# 1. Fetch Indicators
df_life = fetch_world_bank_data("SP.DYN.LE00.IN", "life_expectancy")
df_health = fetch_world_bank_data("SH.XPD.CHEX.PP.CD", "health_expenditure")
df_pop = fetch_world_bank_data("SP.POP.TOTL", "population")
df_gdp = fetch_world_bank_data("NY.GDP.PCAP.PP.CD", "gdp_per_capita")
df_doctors = fetch_world_bank_data("SH.MED.PHYS.ZS", "physicians_per_1000")

# 2. Merge all datasets into one consolidated table
print("Merging datasets...")
df_merged = pd.merge(df_life, df_health, on=["country", "country_code", "year"], how="outer")
df_merged = pd.merge(df_merged, df_pop, on=["country", "country_code", "year"], how="outer")
df_merged = pd.merge(df_merged, df_gdp, on=["country", "country_code", "year"], how="outer")
df_merged = pd.merge(df_merged, df_doctors, on=["country", "country_code", "year"], how="outer")

# 3. Clean up empty placeholder rows
df_clean = df_merged.dropna(subset=["life_expectancy", "health_expenditure", "gdp_per_capita"], how="all")

# 4. Save to SQLite Database
print("Saving to local SQLite database...")
conn = sqlite3.connect("../database/healthcare.db")
df_clean.to_sql("health_metrics", conn, if_exists="replace", index=False)

# 5. Verify the enhanced database schema
print("\n--- Multivariable Ingestion Complete ---")
sample_query = """
SELECT country, year, life_expectancy, health_expenditure, gdp_per_capita, physicians_per_1000 
FROM health_metrics 
WHERE health_expenditure IS NOT NULL AND gdp_per_capita IS NOT NULL 
LIMIT 5;
"""
print(pd.read_sql_query(sample_query, conn))

conn.close()

Fetching life_expectancy...
Fetching health_expenditure...
Fetching population...
Fetching gdp_per_capita...
Fetching physicians_per_1000...
Merging datasets...
Saving to local SQLite database...

--- Multivariable Ingestion Complete ---
       country  year  life_expectancy  health_expenditure  gdp_per_capita  \
0  Afghanistan  2002           56.225           85.857495      926.507941   
1  Afghanistan  2003           57.171           85.933025      966.962032   
2  Afghanistan  2004           57.810           93.935804      971.633503   
3  Afghanistan  2005           58.247          105.927706     1076.087353   
4  Afghanistan  2006           58.553          118.405820     1121.834471   

   physicians_per_1000  
0                  NaN  
1                  NaN  
2                  NaN  
3                  NaN  
4                0.166  
